# P03 · Análisis de operaciones ejecutadas
**QuantLab | Módulo 1 · Fundamentos financieros**

**Pregunta:** ¿qué información aporta el lado agresor de las transacciones sobre la presión de compra y venta?

**Objetivos:** interpretar Trades, calcular volumen base y nominal, VWAP, Trade Imbalance, CVD e intensidad; comparar ventanas temporales y formular una hipótesis sin confundir asociación con predicción.

**Ruta:** ejemplo de cinco operaciones → cálculos → datos recientes de Kraken → agregación temporal → ejercicios. Se puede ejecutar en Colab, JupyterLab o VS Code. Las celdas guiadas y pruebas son ejecutables, y los ejercicios están resueltos en la sección 8.

Las funciones están en **`src/quantlab/p03/trades.py`** y sus pruebas en **`tests/test_trades.py`** (`pytest` desde la raíz del proyecto).

## 1. Preparación
En local, usa el entorno QuantLab. Si faltan dependencias, ejecuta `%pip install pandas numpy matplotlib requests` en una celda. No se necesitan credenciales ni se envían órdenes. El modo inicial es sintético, para reproducir exactamente los cálculos; cambia `USE_LIVE_DATA` a `True` y vuelve a ejecutar desde el principio para consultar Kraken. Una consulta fallida se detiene: no se sustituyen datos reales silenciosamente.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display
from pathlib import Path
import inspect
import json
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p03.trades import (EXAMPLE_TRADES, KRAKEN_TRADES_URL, add_trade_features, aggregate_trades,
                                 compare_windows, count_imbalance, fetch_recent_trades, normalize_trades,
                                 summarize_trades, synthetic_trades, top_volume_windows, trade_metrics)

USE_LIVE_DATA = False
PAIR = 'XBTEUR'
SYMBOL = 'BTC/EUR'
COUNT = 1000
WINDOW = '10s'
EXPORT = False
URL = KRAKEN_TRADES_URL  # 'https://api.kraken.com/0/public/Trades'
plt.rcParams.update({'figure.figsize': (10, 3.5), 'axes.grid': True,
                     'grid.alpha': 0.23, 'axes.spines.top': False,
                     'axes.spines.right': False})

## 2. Anatomía de una operación
Un trade es una ejecución, no una orden pendiente. Cada ejecución tiene dos contrapartes. El campo `side` identifica al agresor (taker), no la totalidad de compradores o vendedores. Una orden limitada también puede ser taker si cruza el libro.

| Campo | Significado | Unidad |
|---|---|---|
| `price` | Precio ejecutado | EUR/BTC |
| `quantity` | Cantidad ejecutada | BTC |
| `timestamp` | Instante del evento | UTC |
| `side` | `b` = compra agresiva; `s` = venta agresiva | categoría |
| `order_type` | `m` = mercado; `l` = limitada | categoría |
| `trade_id` | Identificador por mercado, si está disponible | identificador |

Referencias: Kraken REST Get Recent Trades, Kraken WebSocket v2 Trades (lado del taker), paginación REST. El cursor `last` se conserva como texto; no es una hora de recepción.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿las operaciones tienen sentido?</b>

`normalize_trades` se detiene con un error en vez de seguir con datos malos si:

- La respuesta está vacía o las filas no tienen 6 o 7 campos (esquema inesperado).
- Hay precios o cantidades ≤ 0 o no finitos, timestamps inválidos o un lado distinto de `b`/`s`.
- Hay `trade_id` duplicados: indicaría una captura solapada o repetida.

Después ordena por timestamp de forma estable, para conservar el orden de llegada cuando dos eventos tienen la misma hora.

</div>

In [ ]:
# 🔎 FASE DE VALIDACIÓN: normalize_trades está en src/quantlab/p03/trades.py
print(inspect.getsource(normalize_trades))

example_rows = EXAMPLE_TRADES  # [precio, cantidad, timestamp, lado, tipo, misc, trade_id]
base = pd.Timestamp('2026-01-01T10:00:00Z').timestamp()
example = normalize_trades(example_rows)
display(example)

## 3. Volumen y VWAP
El volumen base suma cantidades; el nominal multiplica cada cantidad por su precio. No se debe sumar directamente los precios ni contar cada contraparte dos veces.

$$V=\sum_i q_i,\qquad N=\sum_i p_iq_i,\qquad VWAP=\frac{N}{V}$$

El VWAP describe el precio medio pagado dentro de la muestra; no es el precio al que necesariamente podemos ejecutar ahora. El precio medio aritmético pondera todas las operaciones por igual.

El volumen con signo vale $+q$ para compras agresivas y $-q$ para ventas agresivas. El desequilibrio toma valores entre −1 y +1:

$$TI=\frac{V_{buy}-V_{sell}}{V_{buy}+V_{sell}}$$

CVD es la suma acumulada de volumen con signo desde el inicio de nuestra muestra; no desde el comienzo del mercado.

In [ ]:
print(inspect.getsource(add_trade_features))
print(inspect.getsource(summarize_trades))

example_features = add_trade_features(example)
display(summarize_trades(example).to_frame('valor'))

## 4. Comprobaciones financieras
Calcula a mano: 1,40 BTC de compras y 0,60 BTC de ventas. El volumen total es 2 BTC. La diferencia es +0,80 BTC y el desequilibrio +0,40. El VWAP debe ser 60.007,15 EUR/BTC.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: comprobamos las funciones con el ejemplo calculado a mano
expected = {'trade_count': 5, 'trade_volume': 2, 'notional_volume': 120014.30,
            'buy_volume': 1.4, 'sell_volume': .6, 'signed_volume': .8,
            'trade_imbalance': .4, 'vwap': 60007.15, 'average_trade_size': .4}
for field, value in expected.items():
    assert np.isclose(summarize_trades(example)[field], value), field
assert np.isclose(example_features['cvd'].iloc[-1], .8)
assert np.isclose(example_features['cumulative_vwap'].iloc[-1], 60007.15)
assert np.isnan(summarize_trades(example.iloc[:0])['trade_imbalance'])
print('12 comprobaciones del ejemplo superadas.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Ejemplo calculado a mano</b>

- Compras agresivas: 0,1 + 0,5 + 0,8 = **1,4 BTC**. Ventas agresivas: 0,2 + 0,4 = **0,6 BTC**.
- Volumen con signo (= CVD final): 1,4 − 0,6 = **+0,8 BTC** → TI = 0,8 / 2 = **+0,40**.
- VWAP = 120.014,30 € / 2 BTC = **60.007,15 €**. Es mayor que la media simple (60.005,20 €) porque la operación más grande (0,8 BTC) se ejecutó al precio más alto (60.010 €).
- Una muestra vacía no tiene imbalance (división entre 0): queda **indefinido** (`NaN`), no 0.

📄 Ver `docs/conceptos/operaciones_ejecutadas.md`

</div>

## 5. Capturar una muestra real
Kraken permite consultar operaciones recientes por REST. Esto es una muestra de eventos, no una serie de intervalos fijos ni una descarga histórica completa. La fecha de recepción del lote es local y distinta de la fecha del trade. Para captura continua usaremos WebSocket más adelante. La opción sintética genera 240 operaciones ficticias y reproducibles, sin ninguna pretensión de modelar el mercado real.

In [ ]:
print(inspect.getsource(fetch_recent_trades))
print(inspect.getsource(synthetic_trades))

trades, provenance = fetch_recent_trades(PAIR, COUNT, SYMBOL) if USE_LIVE_DATA else synthetic_trades(PAIR, SYMBOL)
features = add_trade_features(trades)
print(provenance)
print('Cobertura UTC:', trades.timestamp.min(), '—', trades.timestamp.max())
display(features.head(6))
display(summarize_trades(trades).to_frame('valor').round(6))

## 6. Visualizar precio, VWAP y CVD
El CVD creciente indica saldo agresor comprador positivo durante ese tramo; no garantiza que el precio suba. El precio puede caer pese a compras agresivas por la liquidez pasiva y otros cambios del mercado. El dato de Trades por sí solo no demuestra absorción ni permite calcular OFI del libro completo.

In [ ]:
tag = 'Kraken real' if USE_LIVE_DATA else 'Simulación'
fig, ax = plt.subplots()
ax.plot(features['timestamp'], features['price'], label='Precio ejecutado', color='#32658a')
ax.plot(features['timestamp'], features['cumulative_vwap'], label='VWAP acumulado', color='#b17633', linestyle='--')
ax.set(title=f'{tag} · BTC/EUR: precio y VWAP', ylabel='EUR/BTC', xlabel='Hora UTC')
ax.legend(); fig.autofmt_xdate(); fig.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(features['timestamp'], features['cvd'], color='#32658a')
ax.axhline(0, color='gray', linewidth=1)
ax.set(title=f'{tag} · Delta acumulado de volumen', ylabel='BTC netos (desde primera operación)', xlabel='Hora UTC')
fig.autofmt_xdate(); fig.tight_layout(); plt.show()

## 7. Agregación por intervalos de tiempo
Una fila por operación no equivale a una fila por segundo. Vamos a construir ventanas no solapadas de 10 segundos, con etiqueta de cierre y convención [inicio, fin). Una ventana sin transacciones tiene volumen cero si la captura es completa, pero su VWAP e imbalance son indefinidos. Las ventanas de frontera podrían estar incompletas por tratarse de una muestra reciente de REST. Las marcamos para excluirlas de estudios posteriores. Un hueco en la muestra no demuestra que el exchange no haya negociado.

In [ ]:
print(inspect.getsource(aggregate_trades))

bars = aggregate_trades(trades, WINDOW)
display(bars.head(8).round(5))
# 🔎 FASE DE VALIDACIÓN: la agregación no pierde ni duplica operaciones
assert bars.trade_count.sum() == len(trades)
assert np.isclose(bars.trade_volume.sum(), trades.quantity.sum())
assert np.isclose(bars.signed_volume.sum(), features.signed_volume.sum())
assert bars.trade_imbalance.dropna().between(-1,1).all()
print('4 comprobaciones de agregación superadas.')

In [ ]:
complete = bars.loc[~bars.boundary_window]
fig, ax = plt.subplots()
ax.bar(complete.index, complete.buy_volume, width=pd.Timedelta(WINDOW)*.75,
       label='Compra agresiva', color='#26816d')
ax.bar(complete.index, -complete.sell_volume, width=pd.Timedelta(WINDOW)*.75,
       label='Venta agresiva (signo negativo)', color='#bc6257')
ax.set(title=f'{tag} · Volumen por ventana de {WINDOW} (sin fronteras)', ylabel='BTC', xlabel='Cierre UTC')
ax.legend(); fig.autofmt_xdate(); fig.tight_layout(); plt.show()

## 8. Ejercicios
**E1.** Reescribe una función que reciba un DataFrame y devuelva VWAP y Trade Imbalance sin llamar a `summarize_trades`. Comprueba con las cinco operaciones del ejemplo. ¿Por qué el VWAP es distinto de la media aritmética?

**E2.** Calcula el desequilibrio por número de operaciones en lugar de volumen. En el ejemplo hay 3 compras y 2 ventas. ¿Por qué cambia el resultado? ¿Qué mide cada uno?

**E3.** Compara las ventanas de 5, 10 y 30 segundos. Excluye fronteras y conserva los intervalos vacíos como indefinidos para VWAP e imbalance. ¿Qué escala temporal muestra más valores extremos? ¿Podrían deberse a pocas operaciones?

**E4.** Diseña una tabla con las cinco ventanas completas de mayor volumen. Incluye número de trades, volumen, VWAP, imbalance y retorno contemporáneo si existe un cierre anterior consecutivo. No uses estos retornos para afirmar capacidad predictiva.

**E5.** Activa `USE_LIVE_DATA = True`, ejecuta todo y documenta la cobertura real, el volumen y el VWAP. ¿Existe presión compradora o vendedora en la muestra? ¿Coincide con la evolución del precio? Si falla la conexión, conserva el error e indica que falta evidencia real.

### E1 · VWAP y Trade Imbalance sin `summarize_trades`

In [ ]:
print(inspect.getsource(trade_metrics))

e1 = trade_metrics(example)
display(pd.Series(e1, name='Ejemplo').to_frame())
# 🔎 FASE DE VALIDACIÓN: coincide con el cálculo a mano y con summarize_trades
assert np.isclose(e1['vwap'], 60007.15) and np.isclose(e1['trade_imbalance'], .4)
assert np.isclose(e1['arithmetic_mean_price'], 60005.2)
s = summarize_trades(example)
assert np.isclose(e1['vwap'], s['vwap']) and np.isclose(e1['trade_imbalance'], s['trade_imbalance'])
print('E1 verificado: VWAP = 60.007,15 EUR, TI = +0,40, media simple = 60.005,20 EUR.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · El VWAP pondera por cantidad; la media simple, no</b>

| Operación | Precio | Cantidad | Peso en la media simple | Peso en el VWAP |
|---|---|---|---|---|
| 1 | 60.000 € | 0,1 BTC | 20 % | 5 % |
| 2 | 60.005 € | 0,5 BTC | 20 % | 25 % |
| 3 | 60.003 € | 0,2 BTC | 20 % | 10 % |
| 4 | 60.010 € | 0,8 BTC | 20 % | **40 %** |
| 5 | 60.008 € | 0,4 BTC | 20 % | 20 % |

- **Media simple = 60.005,20 €**: cada operación pesa lo mismo, sea de 0,1 o de 0,8 BTC.
- **VWAP = 60.007,15 €**: cada operación pesa según su cantidad. La operación de 0,8 BTC al precio más alto (60.010 €) pesa el 40 % y tira del VWAP hacia arriba.
- El VWAP responde a "¿qué precio medio se pagó **por cada BTC** negociado?". La media simple responde a "¿qué precio medio tuvo **cada operación**?", y una operación diminuta pesa igual que una enorme.
- Con datos reales la diferencia puede ser grande: en la muestra de Kraken (E5) la media simple fue 57.277,77 € y el VWAP 57.201,19 €.

</div>

### E2 · Desequilibrio por número de operaciones

In [ ]:
print(inspect.getsource(count_imbalance))

e2 = pd.DataFrame({
    'Por volumen (TI)': [summarize_trades(d)['trade_imbalance'] for d in (example, trades)],
    'Por número de operaciones': [count_imbalance(d) for d in (example, trades)],
}, index=['Ejemplo (5 trades)', f'Muestra ({provenance["mode"]})'])
display(e2.round(4))
# 🔎 FASE DE VALIDACIÓN: 3 compras y 2 ventas -> (3 - 2) / 5 = 0,2
assert np.isclose(count_imbalance(example), .2)
print('E2 verificado: desequilibrio por número = +0,20 frente a +0,40 por volumen.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · Contar operaciones no es lo mismo que sumar volumen</b>

- **Por número:** (3 − 2) / 5 = **+0,20**. **Por volumen:** (1,4 − 0,6) / 2 = **+0,40**.
- Cambia porque las compras fueron, de media, más grandes (1,4 / 3 ≈ 0,47 BTC) que las ventas (0,6 / 2 = 0,30 BTC).
- **El TI por volumen** mide cuánta **cantidad** movieron los agresores de cada lado: la presión en BTC.
- **El desequilibrio por número** mide cuántas **decisiones** agresivas hubo de cada lado, sin importar su tamaño. Una sola orden de 10 BTC cuenta igual que una de 0,0001 BTC.
- Si los dos discrepan, el tamaño de las operaciones está desequilibrado. En Kraken (E5): **+0,52 por volumen** y **+0,46 por número**. La mediana de una operación real fue de solo 0,0024 BTC, frente a un máximo de 3,68 BTC. Las órdenes también se trocean en varios trades, así que ninguno de los dos cuenta "personas".

</div>

### E3 · Ventanas de 5, 10 y 30 segundos

In [ ]:
print(inspect.getsource(compare_windows))

EXTREME = 0.8  # |TI| >= 0,8 se considera extremo
e3 = compare_windows(trades, ('5s', '10s', '30s'), extreme=EXTREME)
display(e3.round(3))

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, window in zip(axes, ['5s', '10s', '30s']):
    b = aggregate_trades(trades, window)
    ti = b.loc[~b.boundary_window, 'trade_imbalance'].dropna()  # las vacías quedan fuera (NaN)
    ax.hist(ti, bins=np.linspace(-1, 1, 21), color='#32658a')
    ax.set(title=f'TI por ventana de {window}', xlabel='Trade Imbalance')
axes[0].set_ylabel('Ventanas')
fig.tight_layout(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · Cuanto más corta la ventana, más extremos (y más ruido)</b>

**Muestra sintética** (240 trades):

| Ventana | Ventanas completas | Vacías | Trades por ventana | Desv. típica del TI | % con \|TI\| ≥ 0,8 |
|---|---|---|---|---|---|
| 5 s | 67 | 0 | 3,5 | 0,67 | **35,8 %** |
| 10 s | 33 | 0 | 7,0 | 0,53 | 12,1 % |
| 30 s | 10 | 0 | 21,1 | 0,40 | 0 % |

**Kraken real** (1.000 trades, 19-11-2026 de 17:43 a 18:42 UTC):

| Ventana | Ventanas completas | Vacías | Trades por ventana | Desv. típica del TI | % con \|TI\| ≥ 0,8 | Trades en extremas / resto |
|---|---|---|---|---|---|---|
| 5 s | 711 | **311** | 2,4 | 0,82 | **90,0 %** | 2,2 / 4,6 |
| 10 s | 355 | 60 | 3,3 | 0,79 | 84,7 % | 2,9 / 5,4 |
| 30 s | 118 | 2 | 8,4 | 0,67 | 57,8 % | 8,2 / 8,7 |

- **La escala de 5 s muestra más valores extremos**, en los dos casos.
- **Sí, se deben en gran parte a las pocas operaciones:** con 1 o 2 trades en una ventana, el TI es casi siempre ±1 por construcción. En Kraken, las ventanas extremas de 5 s tenían 2,2 trades de media, frente a 4,6 en el resto.
- Las ventanas vacías (311 de 711 en 5 s) quedan **fuera** del TI y el VWAP. Si las contáramos como TI = 0, falsearíamos la distribución.
- Un TI extremo en una ventana corta **no es una señal fuerte por sí mismo**. Antes de interpretarlo, hay que mirar cuántas operaciones lo sostienen.

</div>

### E4 · Las cinco ventanas completas de mayor volumen

In [ ]:
print(inspect.getsource(top_volume_windows))

e4 = top_volume_windows(bars, 5)
display(e4.round(4))
# 🔎 FASE DE VALIDACIÓN: solo ventanas completas, ordenadas por volumen
assert not bars.loc[e4.index, 'boundary_window'].any()
assert e4['trade_volume'].is_monotonic_decreasing
print('E4: retornos contemporáneos (describen la ventana, no anticipan la siguiente).')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · Volumen alto no implica retorno en la misma dirección</b>

**Kraken real, ventanas de 10 s** (19-11-2026):

| Cierre UTC | Trades | Volumen | VWAP | TI | Retorno contemporáneo |
|---|---|---|---|---|---|
| 18:02:30 | 42 | **16,65 BTC** | 57.120,99 € | +0,58 | **+0,18 bps** |
| 17:55:00 | 38 | 3,69 BTC | 57.182,99 € | +1,00 | +9,76 bps |
| 18:14:20 | 14 | 1,71 BTC | 57.085,52 € | +1,00 | +4,24 bps |
| 18:29:00 | 31 | 1,24 BTC | 57.559,01 € | +0,92 | +16,95 bps |
| 18:20:50 | 5 | 0,91 BTC | 57.342,73 € | +1,00 | +3,47 bps |

- **Una sola ventana concentró el 42 % del volumen de toda la hora** (16,65 de 39,19 BTC). Su TI fue +0,58, pero el precio apenas se movió (+0,18 bps). La liquidez pasiva absorbió la compra agresiva.
- En las cinco ventanas, el TI y el retorno contemporáneo tienen el mismo signo, pero el tamaño del retorno no guarda relación con el volumen.
- El retorno solo se calcula si la ventana anterior es **consecutiva y tuvo operaciones**: si no, queda indefinido.
- Son retornos **contemporáneos**: describen lo que pasó dentro de la misma ventana. No dicen nada sobre la ventana siguiente, que es lo que haría falta para hablar de predicción (sección 9).

</div>

### E5 · Muestra real de Kraken
Cambia `LIVE_E5` a `True` para descargar operaciones recientes y repetir el análisis. Si la conexión falla, la celda se detiene con el error: no hay sustitución por datos ficticios.

In [ ]:
LIVE_E5 = False

if LIVE_E5:
    live, live_prov = fetch_recent_trades(PAIR, COUNT, SYMBOL)  # si falla, lanza el error
    live_summary = summarize_trades(live)
    first, last = live.price.iloc[0], live.price.iloc[-1]
    print('Recepción UTC:', live_prov['received_at'], '| cursor:', live_prov['cursor'])
    print('Cobertura UTC:', live.timestamp.min(), '—', live.timestamp.max(),
          f'({(live.timestamp.max() - live.timestamp.min()).total_seconds() / 60:.1f} min)')
    display(live_summary.to_frame('valor').round(6))
    print(f'Desequilibrio por número: {count_imbalance(live):+.3f}')
    print(f'Precio: {first:,.2f} -> {last:,.2f} EUR ({(last / first - 1) * 10_000:+.2f} bps) '
          f'| mín {live.price.min():,.2f} | máx {live.price.max():,.2f}')
    live_features = add_trade_features(live)
    fig, ax1 = plt.subplots()
    ax1.plot(live_features.timestamp, live_features.price, color='#32658a', label='Precio')
    ax2 = ax1.twinx()
    ax2.plot(live_features.timestamp, live_features.cvd, color='#b17633', label='CVD')
    ax1.set(title='Kraken real · precio y CVD', ylabel='EUR/BTC', xlabel='Hora UTC'); ax2.set_ylabel('CVD (BTC)')
    fig.autofmt_xdate(); fig.tight_layout(); plt.show()
else:
    print('LIVE_E5 = False: no se consulta Kraken. Resultados documentados en la conclusión de abajo.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · Kraken BTC/EUR (19-11-2026, recepción 18:42:42 UTC)</b>

| Métrica | Valor |
|---|---|
| Cobertura | 17:43:21 → 18:42:40 UTC (**59 min**, 1.000 trades) |
| Volumen | **39,19 BTC** (2,24 M€ nominales) |
| Compras / ventas agresivas | 29,73 / 9,46 BTC |
| Trade Imbalance | **+0,52** (por número: +0,46) |
| CVD final | +20,27 BTC |
| VWAP | **57.201,19 €** (media simple: 57.277,77 €) |
| Precio | 57.329,10 → 57.441,60 € (**+19,6 bps**); mín. 57.033 € (18:08), máx. 57.628,10 € (18:28) |
| Tipo de orden | 595 limitadas, 405 de mercado |

- **Sí hubo presión compradora agresiva** en el conjunto de la muestra (TI +0,52), y el precio terminó subiendo (+19,6 bps).
- **Pero la coincidencia no es uniforme:** antes de la ventana grande de las 18:02, el TI ya era +0,40 y aun así el precio **cayó** de 57.329 € a 57.118 € (−37 bps). Después, el TI fue +0,53 y el precio subió. El CVD creciente no garantiza que el precio suba.
- El VWAP (57.201 €) quedó **por debajo** del primer y del último precio porque gran parte del volumen se negoció en el valle de las 18:02–18:08.
- La mayoría de los trades fueron órdenes **limitadas** que cruzaron el libro: el lado agresor no equivale a "orden de mercado".
- Es **una sola muestra de una hora**. No permite afirmar que el desequilibrio anticipe el precio.

</div>

## 9. Primera hipótesis para investigar más adelante
¿Un desequilibrio agresor extremo en una ventana cerrada anticipa el retorno de la siguiente ventana? Necesitaremos retornos futuros de mid price (preferible al último trade para reducir rebote bid-ask), sincronización temporal, muchos días de datos y validación fuera de muestra. La señal solo estará disponible tras el cierre de la ventana y la recepción de sus eventos. Separaremos retornos contemporáneos de futuros y descontaremos costes cuando construyamos una estrategia. No se valida esta hipótesis en P03.

## 10. Exportación opcional e integración
Los datos pueden guardarse para analizarlos después en Jupyter local. Esta exportación no modifica el repositorio existente. Los ficheros se etiquetan por origen; no mezclar ejemplos sintéticos con datos reales. Las funciones `normalize_trades`, `summarize_trades` y `aggregate_trades` ya están integradas en `src/quantlab/p03/trades.py`, con sus pruebas en `tests/test_trades.py`. El prototipo usa floats para análisis educativo; un motor de ejecución necesitaría precisión decimal o unidades discretas.

In [ ]:
if EXPORT:
    folder = Path('quantlab_exports')
    folder.mkdir(exist_ok=True)
    stamp = pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
    prefix = folder / f'p03_{provenance["mode"]}_{stamp}'
    trades.to_csv(str(prefix)+'_trades.csv', index=False)
    bars.to_csv(str(prefix)+'_windows.csv')
    Path(str(prefix)+'_metadata.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
    print('Exportación:', prefix)
else:
    print('Exportación desactivada. Cambia EXPORT a True si quieres guardar la muestra.')

## 11. Conclusiones

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P03</b>

**¿Qué diferencia hay entre liquidez pendiente y volumen ejecutado?**
La liquidez pendiente (P02) son órdenes en el libro que **todavía no se han ejecutado** y que se pueden cancelar. El volumen ejecutado (P03) son operaciones que **ya ocurrieron**: cada una tiene un comprador y un vendedor. El libro dice qué podría pasar; los trades, qué pasó.

**¿Qué mide el Trade Imbalance? ¿Qué no mide?**
Mide qué lado tomó la iniciativa: la diferencia entre el volumen comprado y el vendido **de forma agresiva**, normalizada entre −1 y +1. **No mide** cuántos compradores o vendedores hay (cada trade tiene los dos), ni la liquidez pasiva que absorbió esas órdenes, ni la dirección futura del precio. Además, depende de la ventana: con pocas operaciones tiende a ±1 por construcción.

**¿Qué aporta el VWAP respecto a la media simple?**
Pondera cada precio por la cantidad negociada: es el precio medio **por BTC**, no por operación. Una operación diminuta no pesa lo mismo que una enorme. Sirve como referencia de "precio justo" de un periodo y para evaluar la calidad de una ejecución, pero no es un precio al que se pueda operar ahora.

**¿Qué limitaciones tiene una sola consulta REST?**
Solo trae las últimas N operaciones (1.000 cubrieron 59 minutos), no un intervalo elegido. Las ventanas de los extremos pueden estar incompletas. La hora de recepción es local y no coincide con la de los trades. No hay continuidad entre consultas sin paginar con el cursor `last`, y es una única muestra: no permite generalizar.

**¿Qué cambiarías antes de integrar estas funciones en producción?**
Usar precisión decimal o unidades enteras en lugar de floats. Capturar de forma continua por WebSocket, con paginación y deduplicación por `trade_id` entre lotes. Guardar la procedencia (fuente, recepción, cursor) junto a los datos. Separar las ventanas incompletas y los huecos de captura de las ventanas realmente sin operaciones. Y sincronizar los trades con el libro (P02) para medir absorción y retornos de mid price.

📄 Resumen en `docs/conclusiones/P03_trades.md`

</div>